# V7 Task 2 — local LLM teacher

V7 uses a local 4-bit Qwen3-8B model through MLX. It scores `train.xlsx` only, requires verbatim evidence for every factor, caches every completed row, and evaluates a grouped blend with V3. It does not read `leaderboard.xlsx` or create a submission.

In [ ]:
%pip install -q --upgrade mlx-lm

In [ ]:
from pathlib import Path
import json
import pandas as pd
import v7_llm_factor_teacher as v7

assert Path('train.xlsx').exists()
cfg = v7.V7Config(
    model_name='mlx-community/Qwen3-8B-4bit',
    max_tokens=300,
    require_verbatim_evidence=True,
)
cfg

## Step 1 — score ten posts

The first run downloads about 4.6 GB. This cell then scores ten posts. Results are saved immediately.

In [ ]:
smoke = v7.generate_teacher_cache('train.xlsx', cfg, limit=10)
smoke

## Step 2 — inspect the ten outputs

Check that `parse_error` is empty and that scores/evidence look reasonable before starting the full run.

In [ ]:
cache_path = Path(cfg.output_dir) / cfg.cache_file
records = [json.loads(line) for line in cache_path.read_text().splitlines()]
pd.DataFrame([{
    'row_id': x['row_id'],
    'scores': x['scores'],
    'evidence': x['evidence'],
    'rejected': x['rejected_nonverbatim'],
    'parse_error': x['parse_error'],
    'seconds': x['seconds'],
} for x in records[-10:]])

## Step 3 — complete all training posts

Run this only after the ten-row inspection looks good. It resumes from the cache. Depending on generation speed, the full pass may take several hours. If the kernel stops, rerun this cell; completed rows are skipped.

In [ ]:
full = v7.generate_teacher_cache('train.xlsx', cfg, limit=None)
full

## Step 4 — evaluate and blend with V3

This cell works only after all 1,635 posts are cached. The nested blend is the conservative result.

In [ ]:
metrics = v7.evaluate_teacher('train.xlsx', cfg)
metrics

In [ ]:
print('V3 fixed:', round(metrics['v3_fixed_quota_macro_f1'], 4))
print('Teacher alone:', round(metrics['teacher_fixed_quota_macro_f1'], 4))
print('Nested V3 + teacher:', round(metrics['nested_blend_fixed_quota_macro_f1'], 4))
print('Selected V3 + teacher:', round(metrics['selected_blend_fixed_quota_macro_f1'], 4))
print('Selected calibrated:', round(metrics['selected_blend_calibrated_macro_f1'], 4))
print('Cache diagnostics:', metrics['cache_diagnostics'])

In [ ]:
per_label = pd.read_csv(Path(cfg.output_dir) / 'oof_per_label.csv')
per_label.sort_values('f1')[['factor', 'support', 'teacher_weight', 'teacher_nonzero_rate', 'precision', 'recall', 'f1']]